- 在 PyTorch 中，%timeit（Jupyter 魔法命令）和 torch.utils.benchmark.Timer（Python 脚本级基准测试）都是评估模型性能的强大工具，但它们在适用场景、底层机制和易用性上有显著区别。

结合你之前的 einsum 与 matmul 性能对比需求，以下是这两种工具的深度对比与实战指南：

# 核心区别对比

| 特性 | `%timeit` (Jupyter) | `torch.utils.benchmark.Timer` (PyTorch) |
| :--- | :--- | :--- |
| 运行环境 | 仅限 Jupyter Notebook | 纯 Python 脚本 / Notebook 均可 |
| GPU 同步 | 需手动加 `torch.cuda.synchronize()` | 自动处理，无需手动同步 |
| 预热机制 | 默认自动过滤冷启动开销 | <font color='red'>内置 Warm-up，自动预热 GPU</font> |
| 迭代控制 | 自动寻找最佳循环次数 | 自适应 (`adaptive_autorange`)，自动调整循环次数 |
| 统计指标 | 均值 ± 标准差 | <font color='red'>均值、中位数、标准差、IQR 等</font> |
| 变量传递 | 自动识别当前作用域变量 | 需通过 `globals={}` 显式传入 |

# 实战：对比 matmul vs einsum
假设我们要测试批量矩阵乘法（3D 张量），以下是两种工具的正确写法：

## 方式一：使用 %timeit（适合快速验证）
⚠️ <font color='red'>避坑警告：GPU 上的 CUDA 操作是异步的。</font>
  - <font color='blue'>如果不加同步，%timeit 测的只是 CPU 把指令发给 GPU 的时间，而不是真正的计算时间！</font>

In [ ]:
import torch

# 准备数据
A = torch.randn(32, 512, 256, device="cuda")
B = torch.randn(32, 256, 128, device="cuda")

# ✅ 正确写法：必须包含 CUDA 同步
%timeit torch.matmul(A, B); torch.cuda.synchronize()
%timeit torch.einsum("bik,bkj->bij", A, B); torch.cuda.synchronize()

## 方式二：使用 torch.utils.benchmark.Timer（适合严谨的基准测试）
这是 PyTorch 官方推荐的微基准测试工具，它完美解决了 GPU 异步和预热问题。

In [ ]:
import torch
import torch.utils.benchmark as benchmark

# 1. 准备数据
A = torch.randn(32, 512, 256, device="cuda")
B = torch.randn(32, 256, 128, device="cuda")

# 2. 定义测试函数
def test_matmul(a, b):
    return torch.matmul(a, b)

def test_einsum(a, b):
    return torch.einsum("bik,bkj->bij", a, b)

# 3. 创建 Timer（注意：变量必须通过 globals 传入）
t_matmul = benchmark.Timer(
    stmt='test_matmul(A, B)',
    globals={'test_matmul': test_matmul, 'A': A, 'B': B}
)

t_einsum = benchmark.Timer(
    stmt='test_einsum(A, B)',
    globals={'test_einsum': test_einsum, 'A': A, 'B': B}
)

# 4. 运行自适应基准测试
# 它会自动预热，并循环足够长的时间（默认至少 100ms）以确保结果稳定
print(t_matmul.adaptive_autorange())
print(t_einsum.adaptive_autorange())

## 为什么强烈推荐 torch.utils.benchmark？
如果你在做严肃的性能对比（比如决定生产环境用 matmul 还是 einsum），Timer 有以下三大优势：
1. 自动预热 (Warm-up)：
   - GPU 首次执行 CUDA Kernel 时需要编译和分配显存，第一次运行往往极慢。adaptive_autorange() 会在正式计时前自动执行几次预热，过滤掉冷启动偏差。
2. 自适应循环次数 (Adaptive Autorange)：
   - 对于极快的操作（如小矩阵乘法），%timeit 可能循环几万次；
   - 对于极慢的操作，循环太多次会浪费时间。Timer 会智能调整循环次数，确保总测试时间足够长（默认 >100ms），从而获得稳定的统计数据。
3. 消除异步干扰：
   - **你不需要在每个语句后面手写 torch.cuda.synchronize()**，<font color='red'>Timer 内部已经完美处理了 CPU 与 GPU 的同步问题</font>。
  
## 💡 总结建议
- 日常调试 / Jupyter 随手测一下：
  - 用 %timeit，但千万别忘了加 torch.cuda.synchronize()。
- <font color='red'>写论文 / 生产环境选型 / 复杂模型 Profiling：</font>
  - **使用 torch.utils.benchmark.Timer。它更严谨、更稳定，且代码可以直接复制到 .py 脚本中运行。**